# Module 2: Cross-donor integration, differential expression, and pathways

**Biological question.** After QC, which differences between HuBMAP lung donors reflect cell-type identity or composition rather than technical batch, and which pathways mark those compartments?

**Learning objectives.** On completing this module you will be able to:
- Integrate across donors and assess whether batch correction changed donor mixing (Apply, Analyze)
- Identify differentially expressed genes and enriched pathways, and distinguish composition change from cell-state change (Analyze)

**Bloom level(s).** Apply, Analyze

**Prerequisites.** Module 1 (QC-preprocessed teaching blocks for Donor_1, Donor_2, Donor_3, Donor_4)

**Time.** Instructional: 45 min. Compute (measured): integration ~32 s, DE/pathways ~24 s; peak RSS ~7.6 GB on DE (16 GB recommended; 8 GB minimum).

**Data required.**
| Resource | File | Size | Where it came from |
|---|---|---|---|
| HuBMAP | `module1_*_qc_preprocessed.h5ad` for four teaching blocks | local processed | Module 1 |
| Gene sets | `MSigDB_Hallmark_2020`, `Reactome_2022`, `GO_Biological_Process_2023` GMT | small | `data/genesets/` |

**Run order.** Run the analysis sections in order; the notebook carries state between cells.


## How this module fits the course

Module 1 produced four QC-preprocessed donors. This module puts them in one space and asks what
differs between them, which is the first point in the course where an answer depends on correcting a
technical effect without erasing a biological one.

The notebook concatenates the donors, integrates with Harmony, assigns cell classes, then separates
two things that look alike in a differential expression table: a change in which cells are present,
and a change in what a given cell type is doing. Pathway enrichment follows the same discipline.

Donor and tissue block are confounded in this cohort, one block per donor, so a between-donor
difference cannot be separated from a between-block difference. Read the composition results with
that limit in view.


## 0. Setup

In [ ]:
from pathlib import Path
import os
import sys

MODULE_ROOT = Path.cwd().resolve()
if MODULE_ROOT.name == "notebooks":
    MODULE_ROOT = MODULE_ROOT.parent
if not (MODULE_ROOT / "scripts").is_dir():
    MODULE_ROOT = next(
        (p for p in MODULE_ROOT.parents if (p / "scripts").is_dir()), MODULE_ROOT
    )
if not (MODULE_ROOT / "scripts").is_dir():
    raise FileNotFoundError(
        f"Cannot find scripts/ from cwd={Path.cwd()}. "
        "Open the notebook from the project root or from its notebooks/ folder."
    )
os.chdir(MODULE_ROOT)
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

import time

import pandas as pd
import scanpy as sc

from scripts.common.methods import methods_narrative_module3, methods_table_module3
from scripts.common.paths import ensure_output_dirs, load_config, resolve
from scripts.common.plotting import umap_panel
from scripts.common.runtime import finalize_timing, peak_rss_mb, rss_checkpoint
from scripts.nb03_integration.concatenate import concatenate_donors
from scripts.nb03_integration.compare import (
    composition_weighted_pseudobulk,
    hubmap_marker_matrix,
    max_across_labels_pseudobulk,
    plot_hubmap_marker_heatmap,
)
from scripts.nb03_integration.export import save_module2_integration_outputs
from scripts.nb03_integration.integrate import (
    _AZIMUTH_TO_COARSE,
    donor_composition,
    ensure_cluster_labels,
    harmony_mixing_metrics,
    harmony_silhouette_by_celltype,
    leiden_resolution_sweep,
    post_harmony_label_views,
    recompute_pca_umap,
    run_harmony,
)
from scripts.nb04_de.enrich import run_enrichment_for_groups, run_prerank_gsea
from scripts.nb04_de.export import save_module2_de_outputs
from scripts.nb04_de.load import gene_symbol_series, load_integrated
from scripts.nb04_de.markers import (
    drop_ribosomal_markers,
    filter_markers_for_enrichment,
    groups_below_min_cells,
    known_marker_presence,
    rank_genes_to_frame,
    run_azimuth_focus_de,
    run_rank_genes,
    run_rank_genes_for_prerank,
    wilcoxon_score_ranks_for_prerank,
)
from scripts.nb04_de.plots import (
    plot_enrichment_bar,
    plot_known_marker_dotplot,
    plot_marker_dotplot,
)

cfg = load_config()
ensure_output_dirs(cfg)
fig_dir = resolve(cfg, "outputs_figures")
# Notebook path must stamp the same timing fields as the CLI (Run 2 ships notebooks).
_integ_timing = {
    "_t0": time.perf_counter(),
    "_compute_seconds": None,
    "peak_rss_mb": None,
    "peak_rss_mb_start": peak_rss_mb(),
    "peak_rss_note": (
        "process peak RSS so far (ru_maxrss); monotone across a loop, not per-step"
    ),
}
_de_timing = None
print("Module 2 inputs:", cfg["module2"]["inputs"])
print(methods_narrative_module3(cfg))
display(methods_table_module3(cfg))


## 1. Load, concatenate, and compute the joint embedding

HuBMAP only. Before Harmony we expect donor structure; after Harmony we **measure** mixing rather than assert it.

In [ ]:
adata, donor_summary, shared_ids = concatenate_donors(cfg)
display(donor_summary)
print(f"Shared genes after inner join: {len(shared_ids):,}")
rss_checkpoint("after_concat", adata)
adata = recompute_pca_umap(adata, cfg)
rss_checkpoint("after_pca_umap", adata)
umap_before = umap_panel(
    adata,
    colors=[c for c in ["donor_id", "donor_label"] if c in adata.obs.columns],
    path=fig_dir / "module2_umap_before_harmony.png",
)


## 2. Harmony batch correction and mixing diagnostics

Two cheap metrics: (1) mean fraction of kNN from a *different* donor on `X_pca` vs `X_pca_harmony`; (2) per-`cell_class` silhouette of donor identity. A small change is reported as small.

**How to read them together.** Foreign-donor kNN rising means neighborhoods mix donors more (integration working). Within-class donor silhouette becoming more negative usually means the same thing  -  donor identity is *less* separable inside a cell class  -  not a failure of Harmony. Do not treat the two metrics as a contradiction.

In [ ]:
if cfg["module2"]["harmony"].get("enabled", True):
    adata = run_harmony(adata, cfg)
    adata.obsm["X_umap"] = adata.obsm["X_umap_harmony"]
    rss_checkpoint("after_harmony", adata)

comparison_colors = post_harmony_label_views(adata, cfg)
# Colored by donor, matching the before figure, so the pair is a like-for-like
# read on whether Harmony mixed the donors. The four-panel label view is the
# separate label-comparison figure below.
umap_after = umap_panel(
    adata,
    colors=[c for c in ["donor_id", "donor_label"] if c in adata.obs.columns],
    path=fig_dir / "module2_umap_after_harmony.png",
)
# Four-panel label view: donor, Azimuth label, Leiden, coarse cell class.
umap_label_comparison = umap_panel(
    adata,
    colors=comparison_colors,
    path=fig_dir / "module2_umap_label_comparison.png",
    ncols=2,
)

diag = cfg["module2"].get("diagnostics") or {}
mix = None
sil = None
if diag.get("enabled", True):
    mix = harmony_mixing_metrics(adata)
    sil = harmony_silhouette_by_celltype(adata, label_col="cell_class")
    rss_checkpoint("after_diagnostics", adata)
    display(mix)
    display(sil.head(20))
else:
    print("Skipping Harmony mixing/silhouette (module2.diagnostics.enabled=false)")
    rss_checkpoint("diagnostics_skipped", adata)


## 3. Coarse `cell_class` mapping

Show the mapping dictionary, labels present in the data, and the set difference. Unmatched labels warn by name; `unlabeled` stays its own category. Airway secretory / gland labels map to coarse `epithelial` (not a separate class and not a silent `other`).


In [ ]:
label_col = ensure_cluster_labels(adata, cfg)
present = sorted(adata.obs[label_col].astype(str).unique())
mapped = sorted(_AZIMUTH_TO_COARSE.keys())
missing_from_dict = sorted(set(present) - set(mapped) - {"unlabeled"})
print("Labels present:", present)
print("In mapping dict:", len(mapped))
print("Present but unmapped (should warn at assign time):", missing_from_dict)
print("cell_class counts:\n", adata.obs["cell_class"].value_counts())

sweep = leiden_resolution_sweep(adata, cfg)
display(sweep)
crosstab = donor_composition(adata, label_col=label_col)
display(crosstab)


## 4. Composition, markers, and weighted pseudobulk

Row-z within one resource is legitimate (shared units). Composition-weighted pseudobulk is computed on the **linear** scale (`expm1`); the max-across-types vector is exported as a labeled contrast for Module 4.

In [ ]:
weighted, fracs, _means = composition_weighted_pseudobulk(adata, label_col=label_col)
max_vec = max_across_labels_pseudobulk(adata, label_col=label_col, linear=True)
marker_mat = hubmap_marker_matrix(
    adata, cfg["module2"]["markers"], label_col=label_col, top_n_labels=12
)
heatmap_path = plot_hubmap_marker_heatmap(
    marker_mat, fig_dir / "module2_hubmap_marker_heatmap.png"
)
display(fracs.rename("fraction").to_frame().head(15))
print("Composition-weighted genes:", len(weighted), "| max-across genes:", len(max_vec))

finalize_timing(_integ_timing)
integ_paths = save_module2_integration_outputs(
    cfg,
    adata,
    donor_summary=donor_summary,
    donor_label_crosstab=crosstab,
    composition_weighted=weighted,
    composition_fractions=fracs,
    max_across_labels=max_vec,
    marker_matrix=marker_mat,
    harmony_mixing=mix,
    harmony_silhouette=sil,
    leiden_sweep=sweep,
    figure_paths={
        "umap_before": umap_before,
        "umap_after": umap_after,
        "umap_label_comparison": umap_label_comparison,
        "marker_heatmap": heatmap_path,
    },
    extras={
        "label_col": label_col,
        "cell_class_unmapped": adata.uns.get("cell_class_unmapped_labels", []),
        "cell_class_mapping_n": adata.uns.get("cell_class_mapping_n", {}),
        **{k: v for k, v in _integ_timing.items() if not str(k).startswith("_t")},
    },
)
print(
    "Wrote:",
    integ_paths["h5ad"],
    f"_compute_seconds={_integ_timing.get('_compute_seconds')}",
    f"peak_rss_mb={_integ_timing.get('peak_rss_mb')}",
)


## 5. Differential expression and pathway enrichment

ORA uses local GMTs with `background=` = measured gene symbols (no network). Marker tables use the top-`n_genes` Wilcoxon slice; preranked GSEA uses a **second full-universe** Wilcoxon pass so the ranked list has both tails (NES sign is otherwise meaningless). Groups below `min_cells_per_group` are named, not silently dropped. Ribosomal protein genes are dropped from ORA inputs **and** Wilcoxon prerank lists; translation-adjacent Reactome/GO terms can still appear  -  check marker genes before claiming a pathway story. The auto-interpretation draft does **not** claim biological consistency.


In [ ]:
_de_timing = {
    "_t0": time.perf_counter(),
    "_compute_seconds": None,
    "peak_rss_mb": None,
    "peak_rss_mb_start": peak_rss_mb(),
    "peak_rss_note": (
        "process peak RSS so far (ru_maxrss); monotone across a loop, not per-step"
    ),
}
adata_de, input_path = load_integrated(cfg)
print("DE input:", input_path, adata_de.shape)
groupby = cfg["module2"]["groupby"]
min_cells = int(cfg["module2"]["de"].get("min_cells_per_group", 50))
dropped = groups_below_min_cells(adata_de, groupby, min_cells)
display(dropped)

rank_key = run_rank_genes(adata_de, cfg, groupby=groupby)
markers = drop_ribosomal_markers(rank_genes_to_frame(adata_de, rank_key), cfg)
az_key, az_markers = run_azimuth_focus_de(adata_de, cfg)
if az_key and not az_markers.empty:
    az_markers = drop_ribosomal_markers(az_markers, cfg)
known = known_marker_presence(adata_de, cfg)

fig_paths = {}
p = plot_known_marker_dotplot(
    adata_de, cfg, groupby=groupby, path=fig_dir / "module2_known_marker_dotplot.png"
)
if p:
    fig_paths["known_marker_dotplot"] = p
p = plot_marker_dotplot(
    adata_de, markers, groupby=groupby, path=fig_dir / "module2_marker_dotplot.png"
)
if p:
    fig_paths["marker_dotplot"] = p

markers_for_enr = filter_markers_for_enrichment(markers, cfg)
background = (
    gene_symbol_series(adata_de)
    .astype(str)
    .str.strip()
    .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NAN": pd.NA})
    .dropna()
    .unique()
    .tolist()
)
background = [g for g in background if not str(g).startswith("ENSG")]
print(f"ORA background genes: {len(background):,}")
enrichment = run_enrichment_for_groups(markers_for_enr, cfg, background_genes=background)
p = plot_enrichment_bar(enrichment, fig_dir / "module2_enrichment_summary.png")
if p:
    fig_paths["enrichment_summary"] = p

# Marker table keeps de.n_genes (top slice). Prerank needs both tails: full-universe ranks.
prerank_key = run_rank_genes_for_prerank(adata_de, cfg, groupby=groupby)
prerank_frames = []
for g in dropped.loc[dropped["kept"], "group"].astype(str):
    ranked = wilcoxon_score_ranks_for_prerank(adata_de, prerank_key, g, cfg=cfg)
    if ranked.empty or len(ranked) < 50:
        continue
    pre = run_prerank_gsea(ranked, cfg)
    if pre is not None and not pre.empty:
        pre = pre.copy()
        pre.insert(0, "group", g)
        prerank_frames.append(pre)
prerank_df = pd.concat(prerank_frames, ignore_index=True) if prerank_frames else pd.DataFrame()

finalize_timing(_de_timing)
de_paths = save_module2_de_outputs(
    cfg,
    marker_df=markers,
    enrichment_df=enrichment,
    known_markers_df=known,
    figure_paths=fig_paths,
    prerank_df=prerank_df,
    dropped_groups=dropped,
    extras={
        "rank_key": rank_key,
        "prerank_rank_key": prerank_key,
        "azimuth_focus_key": az_key,
        "n_azimuth_focus_markers": int(az_markers.shape[0]),
        "n_markers_for_enrichment": int(markers_for_enr.shape[0]),
        "n_background_genes": len(background),
        "n_ribosomal_removed_from_marker_table": int(markers.attrs.get("n_ribosomal_removed", 0)),
        "n_ribosomal_removed_before_enrichment": int(
            markers_for_enr.attrs.get("n_ribosomal_removed", 0)
        ),
        "exclude_ribosomal_genes_de": bool(cfg["module2"]["de"].get("exclude_ribosomal_genes", True)),
        "exclude_ribosomal_genes": bool(
            markers_for_enr.attrs.get("exclude_ribosomal_genes", True)
        ),
        **{k: v for k, v in _de_timing.items() if not str(k).startswith("_t")},
    },
)
print(
    "Interpretation:",
    de_paths["interpretation"],
    f"_compute_seconds={_de_timing.get('_compute_seconds')}",
    f"peak_rss_mb={_de_timing.get('peak_rss_mb')}",
)
if "n_input_genes" in enrichment.columns:
    display(enrichment.groupby("group", observed=True)["n_input_genes"].first())


## 6. Composition versus expression

Donor x label crosstab answers "is this population bigger?" Marker / pathway tables answer "are these cells behaving differently?" Keep those questions separate when you write results.

In [ ]:
# Re-display key composition table written above
crosstab_path = resolve(cfg, "outputs_tables") / "module2_donor_by_label_crosstab.tsv"
if crosstab_path.exists():
    display(pd.read_csv(crosstab_path, sep="\t").head())
print("Done.")


## Questions this result raises

1. Which cell population or cluster was analyzed?
2. What are the top marker genes, and do they match known lung identities?
3. What pathways are enriched, and are they biologically coherent for that population?
4. What biological interpretation is supported by markers + pathways together?
5. What limitations remain (label transfer uncertainty, enrichment bias, small input lists)?
6. Did Harmony change donor mixing in a way that matches the silhouette story, or do the two metrics disagree?
7. Where does composition change stop and cell-state change begin in these tables?
